# **Multimodal Models**

**Multimodal learning** trains models that jointly process information from more than one **modality** — a distinct *type* of input signal such as images, text, audio, video, depth, or tabular data. Each modality has its own statistics and natural representation (pixels, tokens, waveforms/spectrograms), and the goal is to combine them so the model understands relationships *across* modalities, e.g. that the word "dog" and a photo of a dog refer to the same concept.

Why bother? Single-modality models are limited by what one signal can express. A caption disambiguates an image; lip motion improves noisy speech recognition; a text prompt steers image generation. Combining modalities yields capabilities no unimodal model has: captioning, visual question answering (VQA), text-to-image generation, and cross-modal retrieval.

## **1. What is a "modality"?**

A modality is a channel of sensory or symbolic information. Common ones in deep learning:

| Modality | Typical raw form | Common encoder |
|---|---|---|
| **Vision** | pixels (H×W×3) | CNN (ResNet) or Vision Transformer (ViT) |
| **Text** | token sequence | Transformer (BERT, GPT) |
| **Audio** | waveform → mel-spectrogram | CNN / audio transformer (Wav2Vec2, AST) |
| **Video** | frames over time | 3D-CNN / spatiotemporal transformer |

The core challenge is the **heterogeneity gap**: different modalities live in different feature spaces with different dimensionalities and statistics. Multimodal architectures exist to *bridge* this gap.

## **2. Fusion strategies**

How and *when* you combine modalities defines the architecture:

- **Early (input) fusion** — concatenate or interleave raw/low-level features before the main network. Maximizes cross-modal interaction but is sensitive to misalignment and missing modalities.
- **Late (decision) fusion** — process each modality with its own network, then combine the final predictions (e.g. average logits, or a small head over concatenated embeddings). Modular and robust, but misses fine-grained interactions.
- **Joint / intermediate fusion** — modality-specific encoders feed a shared model (often via **cross-attention**) that mixes features at intermediate layers. This is what most modern multimodal transformers do, balancing interaction and modularity.

A related axis is **coordinated** vs **joint** representations: coordinated methods (like CLIP) keep separate encoders but *align* their outputs in a shared space, while joint methods fuse into one fused representation.

## **3. Shared embedding spaces and contrastive alignment**

Many multimodal systems learn a **shared embedding space** where semantically related inputs from different modalities land near each other. The dominant recipe is **contrastive alignment** (e.g. CLIP):

- Encode a batch of $N$ image–text pairs into L2-normalized vectors $\{u_i\}$ (images) and $\{v_j\}$ (texts).
- Compute pairwise cosine similarities $s_{ij} = u_i^\top v_j / \tau$, where $\tau$ is a learned temperature.
- Apply a symmetric **InfoNCE** loss: each image's matching text should be its most similar text among the batch, and vice versa.

```python
import torch
import torch.nn.functional as F

def clip_contrastive_loss(image_emb, text_emb, temperature=0.07):
    # image_emb, text_emb: (N, d)
    image_emb = F.normalize(image_emb, dim=-1)
    text_emb = F.normalize(text_emb, dim=-1)
    logits = image_emb @ text_emb.t() / temperature  # (N, N)
    labels = torch.arange(len(logits), device=logits.device)
    loss_i = F.cross_entropy(logits, labels)          # image -> text
    loss_t = F.cross_entropy(logits.t(), labels)      # text -> image
    return (loss_i + loss_t) / 2
```

This single idea — a shared space trained contrastively — powers retrieval, zero-shot classification, and the vision encoders used inside generative VLMs.

## **4. Sub-topics in this section**

| Notebook | Topic |
|---|---|
| [vision_language_models.ipynb](vision_language_models.ipynb) | Vision-Language Models: CLIP-style dual encoders vs fusion/generative VLMs (BLIP-2, LLaVA), captioning and VQA |
| [text_to_image.ipynb](text_to_image.ipynb) | Text-to-image generation: latent diffusion, Stable Diffusion (text encoder + UNet + VAE), classifier-free guidance |
| [audio_visual.ipynb](audio_visual.ipynb) | Audio-visual learning: correspondence, lip-sync, sound localization, AV speech recognition |
| [multimodal_transformers.ipynb](multimodal_transformers.ipynb) | Multimodal transformers: shared token sequences, cross- vs co-attention, ViLBERT / Perceiver |
| [cross_modal_retrieval.ipynb](cross_modal_retrieval.ipynb) | Cross-modal retrieval: contrastive embeddings, image↔text search, recall@K, ANN |

**References**
- Radford et al., *Learning Transferable Visual Models From Natural Language Supervision* (CLIP), 2021.
- Baltrušaitis et al., *Multimodal Machine Learning: A Survey and Taxonomy*, IEEE TPAMI 2019.